In [92]:
from utils import readin_raw, find_fds_for_rhs, fuzzy_filter,turnListToString

In [9]:
fds = readin_raw(path='results/2025-12-01_08-33-55_fds',head_string='country_code,country_name,country_capital_name,country_capital_province,country_area,country_population,city_position,city_name,city_country_code,city_province,city_population,city_latitude,city_longitude,city_elevation,airport_iata_code,airport_name,airport_country_code,airport_city_name,airport_province,airport_island,airport_latitude,airport_longitude,airport_elevation,airport_gmt_offset,language_name,language_percentage')

{'column1': 'country_code', 'column2': 'country_name', 'column3': 'country_capital_name', 'column4': 'country_capital_province', 'column5': 'country_area', 'column6': 'country_population', 'column7': 'city_position', 'column8': 'city_name', 'column9': 'city_country_code', 'column10': 'city_province', 'column11': 'city_population', 'column12': 'city_latitude', 'column13': 'city_longitude', 'column14': 'city_elevation', 'column15': 'airport_iata_code', 'column16': 'airport_name', 'column17': 'airport_country_code', 'column18': 'airport_city_name', 'column19': 'airport_province', 'column20': 'airport_island', 'column21': 'airport_latitude', 'column22': 'airport_longitude', 'column23': 'airport_elevation', 'column24': 'airport_gmt_offset', 'column25': 'language_name', 'column26': 'language_percentage'}


In [23]:
genuine = []
ungenuine = []

In [25]:
genuine.append(fuzzy_filter(find_fds_for_rhs('city_position',fds, sorted=True), include_list=['lat', 'lo'])[1])
ungenuine.append(fuzzy_filter(find_fds_for_rhs('city_position',fds, sorted=True), include_list=['lat', 'lo'])[0])

In [27]:
genuine.append(fuzzy_filter(find_fds_for_rhs('country_population',fds, sorted=True), include_list=['country_code'])[0])

In [29]:
genuine.append(fuzzy_filter(find_fds_for_rhs('language_percentage',fds, sorted=True), include_list=['country_code'])[0])

In [30]:
genuine.append(fuzzy_filter(find_fds_for_rhs('airport_iata_code',fds, sorted=True), exclude_list=[], include_list=['country_code'])[0])

In [31]:
genuine

[(['city_latitude', 'city_longitude'], 'city_position'),
 (['country_code'], 'country_population'),
 (['country_code', 'language_name'], 'language_percentage'),
 (['country_code', 'airport_name', 'city_name'], 'airport_iata_code')]

In [ ]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('city_position',fds, sorted=True), exclude_list= ['lat'], include_list=['lo']))

In [37]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('city_position',fds, sorted=True), exclude_list= ['lo'], include_list=['lat'])[:5])

In [ ]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('country_population',fds, sorted=True), exclude_list=['code'])[:3])

[(['airport_latitude'], 'country_population'),
 (['airport_longitude'], 'country_population'),
 (['city_position'], 'country_population')]

In [43]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('country_population',fds, sorted=True), exclude_list=['code'])[4:11])

In [48]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('language_percentage',fds, sorted=True), exclude_list=['_code'])[:2])

In [67]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('language_percentage',fds, sorted=True), exclude_list=['_code'])[5:10])

In [77]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('airport_iata_code',fds, sorted=True), exclude_list=[], include_list=[])[:2])

In [76]:
ungenuine.extend(fuzzy_filter(find_fds_for_rhs('airport_iata_code',fds, sorted=True), exclude_list=[], include_list=[])[3:11])

In [79]:
len(ungenuine)

38

In [ ]:
ungenuine.append(fuzzy_filter(fds, exclude_list=['airport_iata_code'], include_list=[])[1])

In [91]:
ungenuine.append(fuzzy_filter(fds, exclude_list=['airport_iata_code','airport_latitude','airport_longitude','city_position', 'code','name'], include_list=[])[3])

In [ ]:
import pandas as pd
def convertToDataFrame(fds, table_name, is_approx, null_eq_null, is_genuine):
    n = len(fds)

    lhs, rhs = zip(*fds)  # unpack pairs

    wb_dataframe = pd.DataFrame({
        "lhs": [turnListToString(x) for x in lhs],
        "rhs": list(rhs),
        "tablename": [table_name + ".csv"] * n,
        "is_approx": [is_approx] * n,
        "null_eq_nul": [null_eq_null] * n,
        "is_genuine": [is_genuine] * n,
    })   
    return wb_dataframe

In [105]:
tablename= 'mondialjoin'
genuine_df = convertToDataFrame(fds=genuine,table_name=tablename, is_approx=False, null_eq_null=True, is_genuine=True )
ungenuine_df = convertToDataFrame(fds=ungenuine,table_name=tablename, is_approx=False, null_eq_null=True, is_genuine=False )

In [111]:
training_df = pd.concat([genuine_df, ungenuine_df], ignore_index=True)

In [113]:
training_df

,lhs,rhs,tablename,is_approx,null_eq_nul,is_genuine
0,"city_latitude, city_longitude",city_position,mondialjoin.csv,False,True,True
1,country_code,country_population,mondialjoin.csv,False,True,True
2,"country_code, language_name",language_percentage,mondialjoin.csv,False,True,True
3,"country_code, airport_name, city_name",airport_iata_code,mondialjoin.csv,False,True,True
4,"city_population, city_longitude",city_position,mondialjoin.csv,False,True,False
5,airport_longitude,city_position,mondialjoin.csv,False,True,False
6,"city_longitude, airport_name",city_position,mondialjoin.csv,False,True,False
7,"city_longitude, city_name",city_position,mondialjoin.csv,False,True,False
8,"city_longitude, airport_city_name",city_position,mondialjoin.csv,False,True,False
9,"city_province, city_longitude",city_position,mondialjoin.csv,False,True,False


In [118]:
import os
import pandas as pd

_FILENAME_TRAININGDATA = 'dataset.csv'

def writeBackTrainingData(df: pd.DataFrame):
    """
    Writes or appends training data to training.csv.

    If training.csv exists:
        - Read it in
        - If the only difference is that the new df is missing an 'id' column:
            * Add an 'id' column to the new df, continuing from the existing max id
        - Check that columns now match
        - Concatenate old + new data
        - Overwrite training.csv with the concatenated dataframe
    If training.csv does not exist:
        - Write df to training.csv
    """

    file_path = os.path.join('.', _FILENAME_TRAININGDATA)

    if os.path.exists(file_path):
        try:
            existing_df = pd.read_csv(file_path)
        except Exception as e:
            raise RuntimeError(f"Failed to read existing {_FILENAME_TRAININGDATA}: {e}")

        new_df = df.copy()

        # ---- Handle special case: existing has 'id', new df doesn't ----
        if 'id' in existing_df.columns and 'id' not in new_df.columns:
            existing_without_id = [c for c in existing_df.columns if c != 'id']

            # Check that all non-id columns match exactly
            if existing_without_id == list(new_df.columns):
                # Determine starting id: continue from max existing id if numeric
                if pd.api.types.is_integer_dtype(existing_df['id']):
                    start_id = int(existing_df['id'].max()) + 1
                else:
                    # Fallback: just continue from length (string or other dtype)
                    start_id = len(existing_df)

                # Use pandas to create the id column
                new_df = new_df.reset_index(drop=True)
                new_df.insert(0, 'id', range(start_id, start_id + len(new_df)))
            else:
                # Columns don't match aside from id → real mismatch
                raise ValueError(
                    f"Column mismatch (ignoring 'id'): existing file has {existing_without_id}, "
                    f"but new data has {list(new_df.columns)}."
                )

        # ---- Final schema check: must match exactly now ----
        if list(existing_df.columns) != list(new_df.columns):
            raise ValueError(
                f"Column mismatch: existing file has {list(existing_df.columns)}, "
                f"but new data has {list(new_df.columns)}."
            )

        # Concatenate and write back
        try:
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        except Exception as e:
            raise RuntimeError(f"Failed to concatenate dataframes: {e}")

        combined_df.to_csv(file_path, index=False)

    else:
        # No existing file: if df has no id and you want one, you can add it here too
        df.to_csv(file_path, index=False)


In [119]:
writeBackTrainingData(training_df)